In [1]:
import numpy as np
import polars as pl


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time

import pandas as pd
from extinction import fitzpatrick99


from scipy.stats import kurtosis, skew, spearmanr, linregress

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

https://github.com/maymeridian/Tidal-Disruption-Classification/blob/competition_winner/src/data_loader.py#L76

## Process:
    1. GET DATA
    
        X_train, y_train = get_prepared_dataset('train')

    2. RUN TRAINING (Using 5-Fold CV + Class Weights + Auto-Tuning)
    
        model, score, threshold = train_with_cv(model_name, X_train, y_train)


 get_prepared_Dataset:
 
     - load_lightcurves (.csv files one row = one obs (with source_id col) I think; should be all combined)
     
     - extract_features:
     
       x log_df = get_log_data(dataset_type) (get pandas DataFrame from csv)
       
       x lc_df = apply_deextinction(lc_df, log_df)
       
       * lc_clean = apply_quality_cuts(lc_df)
       
       * Fit 2D GP and get GP features: get_gp_features
       
       * if Z: additional set of features


   
df vs log_df? (log_df has target / labels from MALLORN I think)

In [2]:
parq = pl.read_parquet('/scratch/gcontard/ELASTICC2/mallorn_like/training_ELAsTiCC2_subtrain_100pos_3000neg_nounsup.parq')
parq

ObjectId,IAUC,FAKE,RA,DEC,PIXSIZE,NXPIX,NYPIX,SNTYPE,NOBS,PTROBS_MIN,PTROBS_MAX,MWEBV,MWEBV_ERR,REDSHIFT_HELIO,REDSHIFT_HELIO_ERR,REDSHIFT_FINAL,REDSHIFT_FINAL_ERR,VPEC,VPEC_ERR,HOSTGAL_NMATCH,HOSTGAL_NMATCH2,HOSTGAL_OBJID,HOSTGAL_FLAG,HOSTGAL_PHOTOZ,HOSTGAL_PHOTOZ_ERR,HOSTGAL_SPECZ,HOSTGAL_SPECZ_ERR,HOSTGAL_RA,HOSTGAL_DEC,HOSTGAL_SNSEP,HOSTGAL_DDLR,HOSTGAL_CONFUSION,HOSTGAL_LOGMASS,HOSTGAL_LOGMASS_ERR,HOSTGAL_LOGSFR,HOSTGAL_LOGSFR_ERR,…,SKY_SIG,SKY_SIG_T,RDNOISE,ZEROPT,ZEROPT_ERR,GAIN,XPIX,YPIX,SIM_FLUXCAL_HOSTERR,SIM_MAGOBS,ELASTICC_class,AGN_PARAM(M_BH),AGN_PARAM(Mi),AGN_PARAM(edd_ratio),AGN_PARAM(edd_ratio2),AGN_PARAM(t_transition),AGN_PARAM(cl_flag),SIM_TEMPLATEMAG_u,SIM_TEMPLATEMAG_g,SIM_TEMPLATEMAG_r,SIM_TEMPLATEMAG_i,SIM_TEMPLATEMAG_z,SIM_TEMPLATEMAG_Y,SIM_HOSTLIB(g_obs),SIM_HOSTLIB(r_obs),SIM_HOSTLIB(i_obs),SIM_HOSTLIB(LOGMASS_TRUE),SIM_HOSTLIB(LOG_SFR),SIM_SALT2x0,SIM_SALT2x1,SIM_SALT2c,SIM_SALT2mB,SIM_SALT2alpha,SIM_SALT2beta,SIM_SALT2gammaDM,spectro_class,binary_class
i64,binary,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,list[f32],list[f32],list[f32],list[f32],list[f32],list[f32],list[f32],list[f32],list[f32],list[f32],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i32
64073152,"b""NULL\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20""",2,303.534237,-14.883878,0.2,-9,-9,126,188,145802,145989,0.07569,0.003784,0.863515,0.84655,0.862012,0.84655,0.0,300.0,1,1,9814193692,0,0.863515,0.84655,-9.0,-9.0,303.534521,-14.883795,1.033465,1.474955,-99.0,10.9037,-9999.0,-9999.0,-9999.0,…,"[80.57, 47.459999, … 52.790001]","[0.0, 0.0, … 0.0]","[0.25, 0.25, … 0.25]","[31.0, 30.110001, … 31.57]","[0.005, 0.005, … 0.005]","[1.0, 1.0, … 1.0]","[-9.0, -9.0, … -9.0]","[-9.0, -9.0, … -9.0]","[0.0, 0.0, … 0.0]","[97.953285, 97.953285, … 29.575298]","""SNIc+HostXT_V19""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,10.9037,-0.3545,null,null,null,null,null,null,null,"""SNIc+HostXT_V19""",0
24795017,"b""NULL\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20""",2,0.289052,-2.730808,0.2,-9,-9,142,263,285529,285791,0.031414,0.001571,2.471862,2.47294,2.471862,2.47294,0.0,300.0,1,1,8937516062,0,2.471862,2.47294,-9.0,-9.0,0.2891,-2.730922,0.444471,0.723441,-99.0,8.1505,-9999.0,-9999.0,-9999.0,…,"[50.470001, 66.059998, … 16.700001]","[0.0, 0.0, … 0.0]","[0.25, 0.25, … 0.25]","[30.110001, 31.389999, … 31.450001]","[0.005, 0.005, … 0.005]","[1.0, 1.0, … 1.0]","[-9.0, -9.0, … -9.0]","[-9.0, -9.0, … -9.0]","[0.0, 0.0, … 0.0]","[99.0, 99.0, … 29.605356]","""TDE""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,8.1505,-4.2499,null,null,null,null,null,null,null,"""TDE""",1
49525253,"b""NULL\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20""",2,196.703556,-29.445324,0.2,-9,-9,131,88,224312,224399,0.079628,0.003981,0.554516,0.54518,0.556084,0.54518,0.0,300.0,1,1,5875306555,0,0.554516,0.54518,-9.0,-9.0,196.703536,-29.445327,0.062235,0.108174,-99.0,9.05454,-9999.0,-9999.0,-9999.0,…,"[49.400002, 47.950001, … 49.59]","[0.0, 0.0, … 0.0]","[0.25, 0.25, … 0.25]","[30.040001, 30.09, … 29.959999]","[0.005, 0.005, … 0.005]","[1.0, 1.0, … 1.0]","[-9.0, -9.0, … -9.0]","[-9.0, -9.0, … -9.0]","[0.0, 0.0, … 0.0]","[97.219139, 97.219139, … 27.75194]","""SNII-Templates""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9.05454,-0.3881,null,null,null,null,null,null,null,"""SNII-Templates""",0
2162176,"b""NULL\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20\x20""",2,127.544521,-25.663591,0.2,-9,-9,83,274,72216,72489,0.093514,0.004676,-9.0,-9.0,-9.0,-9.0,0.0,0.0,0,0,0,0,-9.0,-9.0,-9.0,-9.0,-999.0,-999.0,-9.0,-9.0,-99.0,-9999.0,-9999.0,-9999.0,-9999.0,…,"[47.77, 46.98, … 46.16]","[0.0, 0.0, … 0.0]","[0.25, 0.25, … 0.25]","[30.120001, 30.129999, … 30.139999]","[0.005, 0.005, … 0.005]","[1.0, 1.0, … 1.0]","[-9.0, -9.0, … -9.0]","[-9.0, -9.0, … -9.0]",

In [3]:
parq_mall = pl.read_parquet('/scratch/gcontard/MALLORN/train_mallorn_data.parq')
parq_mall

FLUXCAL,FLUXCALERR,MJD,BAND,spectro_class,binary_class,EBV,Z,Z_err,ObjectId
list[f64],list[f64],list[f64],list[str],str,i64,f64,f64,f64,str
"[-1.970311, 0.833014, … 0.412255]","[0.807963, 0.368413, … 0.125994]","[61460.8727, 63968.1111, … 62948.3276]","[""y"", ""i"", … ""r""]","""AGN""",0,0.032,1.198,NaN,"""Dornhoth_anwar_melethron"""
"[0.274922, 0.191174, … 0.405841]","[2.195014, 0.139372, … 0.101548]","[62913.1184, 63958.2391, … 63528.3593]","[""z"", ""g"", … ""g""]","""AGN""",0,0.016,0.226,NaN,"""Dornhoth_archam_grond"""
"[0.189548, -0.022459, … 1.374008]","[0.202875, 0.2248284, … 0.58815]","[62354.8439, 63855.7029, … 63111.5374]","[""r"", ""g"", … ""z""]","""SN Ia""",0,0.011,0.4052,NaN,"""Dornhoth_certh_iaun"""
"[1.051772, 0.290583, … 2.107653]","[0.8639606, 0.1897211, … 0.200777]","[64025.3964, 63712.8062, … 63329.707]","[""y"", ""r"", … ""r""]","""SN Ia""",0,0.132,0.2748,NaN,"""Dornhoth_drafn_celon"""
"[-1.630159, 10.499389, … 0.802379]","[0.365777, 0.253867, … 0.186741]","[63314.4662, 63780.9674, … 63640.1368]","[""z"", ""r"", … ""i""]","""AGN""",0,0.11,3.049,NaN,"""Dornhoth_fervain_onodrim"""
…,…,…,…,…,…,…,…,…,…
"[-0.196774, 0.079269, … 0.471519]","[0.493123, 0.145988, … 0.416845]","[63990.7231, 64005.6823, … 64449.4725]","[""z"", ""r"", … ""z""]","""SN Ia""",0,0.036,0.3925,NaN,"""yll_lebenedh_cair"""
"[2.115769, 2.552514, … -0.58304]","[0.334359, 0.450688, … 1.334266]","[63774.3224, 63774.3224, … 64529.8902]","[""r"", ""i"", … ""y""]","""TDE""",1,0.045,1.701,NaN,"""yll_merilin_rach"""
"[1.258772, 1.62227, … 0.156546]","[0.195577, 0.400587, … 0.13018]","[63971.4577, 63971.4577, … 64624.1071]","[""i"", ""z"", … ""r""]","""SN Ia""",0,0.134,0.365,NaN,"""yll_minai_gondrath"""


In [4]:
# Effective wavelengths (Angstroms) for LSST filters
FILTER_WAVELENGTHS = {
    'u': 3641,
    'g': 4704,
    'r': 6155,
    'i': 7504,
    'z': 8695,
    'y': 10056,    
    'Y': 10056
}

def apply_deextinction_parq(df):
    if 'Flux_Corrected' in df.columns: 
        return df
    if 'EBV' not in df.columns and 'MWEBV' not in df.columns:    
        df = df.with_columns([
            pl.col('FLUXCAL').alias('Flux_Corrected'),
            pl.col('FLUXCALERR').alias('Flux_err_Corrected')
        ])
        
        return df
    # ebv_map = log_df.set_index('object_id')['EBV']
    # df['EBV'] = df['object_id'].map(ebv_map)

    ebv_key = 'EBV' if 'EBV' in df.columns else 'MWEBV'
    
    unique_filters = list(FILTER_WAVELENGTHS.keys())
    unique_wls = np.array([FILTER_WAVELENGTHS[f] for f in unique_filters], dtype=float)
    ext_factors = fitzpatrick99(unique_wls, 1.0)
    ext_map = dict(zip(unique_filters, ext_factors))

    def correct_flux(fluxes, bands, ebv):
        a_lambda = np.array([ext_map[b] for b in bands]) * (ebv * 3.1)
        correction = 10 ** (a_lambda / 2.5)
        return (np.array(fluxes) * correction).tolist()

    df = df.with_columns([
        pl.struct(['FLUXCAL', 'BAND', ebv_key]).map_elements(
            lambda r: correct_flux(r['FLUXCAL'], r['BAND'], r[ebv_key]),
            return_dtype=pl.List(pl.Float64)
        ).alias('Flux_Corrected'),

        pl.struct(['FLUXCALERR', 'BAND', ebv_key]).map_elements(
            lambda r: correct_flux(r['FLUXCALERR'], r['BAND'], r[ebv_key]),
            return_dtype=pl.List(pl.Float64)
        ).alias('Flux_err_Corrected'),
    ])
    
    return df


In [5]:
def apply_quality_cuts(lc_df):
    # if 'SNR' not in lc_df.columns:
    #     safe_err = lc_df['Flux_err'].replace(0, 1e-5)
    #     lc_df['SNR'] = lc_df['Flux'] / safe_err
    ## I don't think they use this anywhere?

    # valid_mask = (lc_df['Flux'] > 0) 
    # counts = lc_df[valid_mask].groupby('object_id').size()
    # keep_ids = counts[counts >= 2].index
    # return lc_df[lc_df['object_id'].isin(keep_ids)].copy()

    ## This is basically removing/ignoring light-curves that have less than 2 positive FLUXCAL
    return lc_df.filter(pl.col('FLUXCAL').list.eval(pl.element() > 0).list.sum() >= 2)

In [6]:
mall_deext = apply_deextinction_parq(parq_mall)
ela_deext = apply_deextinction_parq(parq)
print(len(mall_deext))
print(len(ela_deext))
mall_deext = apply_quality_cuts(mall_deext)
ela_deext = apply_quality_cuts(ela_deext)
print('--"quality cut"--')
print(len(mall_deext))
print(len(ela_deext))

3043
2337
--"quality cut"--
3043
2335


In [7]:
obj = ela_deext.filter(pl.col('ObjectId')==64073152)
obj['Flux_Corrected'][0].max()

113.92573582609913

In [8]:
def fit_2d_gp(obj_df):
    ## Assume a one row pl.DF for one obj 
    
    if 'Flux_Corrected' in obj_df.columns:
        y = obj_df['Flux_Corrected'][0].to_numpy()
        y_err = obj_df['Flux_err_Corrected'][0].to_numpy()
    else:
        y = obj_df['FLUXCAL'][0].to_numpy()
        y_err = obj_df['FLUXCALERR'][0].to_numpy()

    X = np.column_stack([obj_df['MJD'][0].to_numpy(),
                         np.array([FILTER_WAVELENGTHS[b] for b in obj_df['BAND'][0]])])

    y_scale = np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else 1.0
    y_norm = y / y_scale
    y_err_norm = y_err / y_scale

    kernel = ConstantKernel(1.0) * Matern(length_scale=[100, 6000], nu=1.5)
    gp = GaussianProcessRegressor(kernel=kernel, alpha=y_err_norm**2, n_restarts_optimizer=0, random_state=42)
    gp.fit(X, y_norm)
    return gp, y_scale


In [36]:


def calculate_template_matching(t_grid, y_pred_g, peak_idx, peak_time, peak_flux):
    """Normalized TDE Shape Matching ONLY."""
    post_peak_indices = np.where(t_grid > peak_time)[0]
    match_tde = 10.0 
    
    if len(post_peak_indices) > 5 and peak_flux > 0:
        y_fade = y_pred_g[post_peak_indices]
        t_fade = t_grid[post_peak_indices] - peak_time
        
        y_norm = y_fade / peak_flux
        half_idx = np.argmax(y_norm < 0.5)
        
        if half_idx > 0:
            t_half = t_fade[half_idx]
            if t_half > 0.1: 
                t_norm = t_fade / t_half 
                y_temp_tde = 1.0 / (1.0 + (t_norm * (2**(1/1.67) - 1)))**1.67
                mask = t_norm < 3.0
                if mask.sum() > 2:
                    match_tde = np.sqrt(np.mean((y_norm[mask] - y_temp_tde[mask])**2))
                    
    return match_tde

def calculate_physics_wars(t_grid, y_pred_g, peak_idx, peak_time, peak_flux):
    # 1. TDE FIT
    post_peak_indices = np.where(t_grid > peak_time)[0]
    tde_error = 10.0 
    linear_decay_slope = 0.0 
    
    if len(post_peak_indices) > 5 and peak_flux > 0:
        y_real_fade = y_pred_g[post_peak_indices]
        t_fade = t_grid[post_peak_indices]
        dt = (t_fade - peak_time) + 10 
        
        y_ideal_tde = peak_flux * (dt / dt[0])**(-1.67)
        residuals_tde = (y_real_fade - y_ideal_tde) / peak_flux
        tde_error = np.mean(residuals_tde**2)

        try:
            log_t = np.log(dt)
            log_y = np.log(y_real_fade + 1e-9)
            # slope, _, _, _, _ = linregress(log_t, log_y)
            # linear_decay_slope = slope
            linear_decay_slope = slope if not np.isnan(slope) else 0.0
        except Exception:
            linear_decay_slope = 0.0

    # 2. RISE PHYSICS
    pre_peak_indices = np.where(t_grid < peak_time)[0]
    rise_fireball_error = 10.0
    pre_peak_var = 0.0 
    
    if len(pre_peak_indices) > 5 and peak_flux > 0:
        y_real_rise = y_pred_g[pre_peak_indices]
        t_rise = t_grid[pre_peak_indices]
        if len(t_rise) > 3:
            try:
                coeffs = np.polyfit(t_rise, y_real_rise, 2)
                p = np.poly1d(coeffs)
                residuals_rise = (y_real_rise - p(t_rise)) / peak_flux
                rise_fireball_error = np.mean(residuals_rise**2)
                pre_peak_var = np.var(residuals_rise)
            except Exception:
                pass

    fade_correlation = 0.0
    # if len(post_peak_indices) > 2:
    #     fade_correlation = np.corrcoef(t_grid[post_peak_indices], y_pred_g[post_peak_indices])[0, 1]
    if len(post_peak_indices) > 2:
        t_post = t_grid[post_peak_indices]
        y_post = y_pred_g[post_peak_indices]
        if np.std(y_post) > 0 and np.std(t_post) > 0:
            fade_correlation = np.corrcoef(t_post, y_post)[0, 1]
        else:
            fade_correlation = 0.0
            
    half_max = peak_flux / 2.0
    rise_idx_candidates = np.where((y_pred_g[:peak_idx] <= half_max))[0]
    t_half_rise = t_grid[rise_idx_candidates[-1]] if len(rise_idx_candidates) > 0 else t_grid[0]
    fade_idx_candidates = np.where((y_pred_g[peak_idx:] <= half_max))[0]
    t_half_fade = t_grid[peak_idx + fade_idx_candidates[0]] if len(fade_idx_candidates) > 0 else t_grid[-1]
    fwhm = t_half_fade - t_half_rise
    
    return tde_error, linear_decay_slope, rise_fireball_error, fade_correlation, fwhm, pre_peak_var


def get_gp_features(obj_id, obj_df):
    try:
        gp, y_scale = fit_2d_gp(obj_df)
    except Exception:
        return None

    params = gp.kernel_.get_params()
    try:
        ls_time = params.get('k2__length_scale', [0,0])[0]
        ls_wave = params.get('k2__length_scale', [0,0])[1]
        amplitude = np.sqrt(params.get('k1__constant_value', 0)) * y_scale
    except Exception:
        ls_time, ls_wave, amplitude = 0, 0, 0

    if 'Flux_Corrected' in obj_df.columns:
        flux_data = obj_df['Flux_Corrected'][0].to_numpy()
        flux_err = obj_df['Flux_err_Corrected'][0].to_numpy()
    else:
        flux_data = obj_df['FLUXCAL'][0].to_numpy()
        flux_err = obj_df['FLUXCALERR'][0].to_numpy()

    significant_negative = (flux_data < -3 * flux_err)
    negative_flux_fraction = significant_negative.sum() / len(flux_data) if len(flux_data) > 0 else 0.0

    significant_mask = flux_data > (3 * flux_err) 

    mjd_obj = obj_df['MJD'][0].to_numpy()
    band_obj = obj_df['BAND'][0].to_numpy()
    valband_obj = np.array([FILTER_WAVELENGTHS[b] for b in band_obj])
    
    detection_times = mjd_obj[significant_mask]
    
    total_survey_span = obj_df['MJD'][0].max() - obj_df['MJD'][0].min()
    
    if len(detection_times) > 4:
        t_10 = np.percentile(detection_times, 10)
        t_90 = np.percentile(detection_times, 90)
        robust_duration = t_90 - t_10
        duty_cycle = robust_duration / total_survey_span if total_survey_span > 0 else 0
    else:
        robust_duration = 0.0
        duty_cycle = 0.0

    flux_kurtosis = kurtosis(flux_data, fisher=True)
    flux_skew = skew(flux_data) 

    X_obs = np.column_stack([mjd_obj, valband_obj])
    y_gp_pred = gp.predict(X_obs) * y_scale
    safe_err = np.where(flux_err <= 0, 1e-5, flux_err)
    chi_sq_terms = ((flux_data - y_gp_pred) / safe_err) ** 2
    reduced_chi_square = np.mean(chi_sq_terms)
    reduced_chi_square = min(reduced_chi_square, 1000.0)

    # 100 Points Grid
    t_min, t_max = mjd_obj.min(), mjd_obj.max()
    t_grid = np.linspace(t_min, t_max, 100)
    g_wave = FILTER_WAVELENGTHS['g']
    X_pred_g = np.column_stack([t_grid, np.full_like(t_grid, g_wave)])
    y_pred_g = gp.predict(X_pred_g) * y_scale

    peak_idx = np.argmax(y_pred_g)
    peak_time = t_grid[peak_idx]
    peak_flux = y_pred_g[peak_idx]
    threshold = peak_flux / 2.512

    # --- 3. PERCENTILE RATIOS ---
    positive_flux = y_pred_g[y_pred_g > 0]
    
    if len(positive_flux) > 0:
        perc_20 = np.percentile(positive_flux, 20)
        perc_50 = np.percentile(positive_flux, 50)
        perc_80 = np.percentile(positive_flux, 80)
        percentile_ratio_20_50 = perc_20 / (perc_50 + 1e-9)
        percentile_ratio_80_max = perc_80 / (peak_flux + 1e-9)
    else:
        percentile_ratio_20_50 = 0.0
        percentile_ratio_80_max = 0.0

    pre_peak = y_pred_g[:peak_idx]
    t_pre = t_grid[:peak_idx]

    if len(pre_peak) > 0 and np.any(pre_peak < threshold):
        drop_idx = np.where(pre_peak < threshold)[0][-1]
        rise_time = peak_time - t_pre[drop_idx]
    else:
        rise_time = peak_time - t_min

    post_peak = y_pred_g[peak_idx:]
    t_post = t_grid[peak_idx:]

    if len(post_peak) > 0 and np.any(post_peak < threshold):
        drop_idx = np.where(post_peak < threshold)[0][0]
        fade_time = t_post[drop_idx] - peak_time
    else:
        fade_time = t_max - peak_time

    # calculate physics
    tde_error, linear_decay_slope, rise_error, fade_shape, fwhm, pre_peak_var = calculate_physics_wars(t_grid, y_pred_g, peak_idx, peak_time, peak_flux)
    match_tde = calculate_template_matching(t_grid, y_pred_g, peak_idx, peak_time, peak_flux)

    def get_val(t, band):
        val = gp.predict([[t, FILTER_WAVELENGTHS[band]]])[0] * y_scale
        return val if val > 0 else 1e-5

    val_u = get_val(peak_time, 'u')
    val_g = get_val(peak_time, 'g')
    val_r = get_val(peak_time, 'r')
    
    flux_ratio_ug = val_u / val_g 
    flux_ratio_gr = val_g / val_r 

    ug_peak = -2.5 * np.log10(val_u / val_g)
    gr_peak = -2.5 * np.log10(val_g / val_r)
    ur_peak = -2.5 * np.log10(val_u / val_r)

    # COLOR FEATURES
    t_samples = np.linspace(peak_time, peak_time + fade_time, 5)
    g_samples = [get_val(t, 'g') for t in t_samples]
    r_samples = [get_val(t, 'r') for t in t_samples]
    gr_colors = [-2.5 * np.log10(g/r) for g, r in zip(g_samples, r_samples)]
    
    mean_color_gr = np.mean(gr_colors)
    std_color_gr = np.std(gr_colors)
    
    color_monotonicity, _ = spearmanr(np.arange(5), gr_colors)
    if np.isnan(color_monotonicity):
         color_monotonicity = 0.0
    
    try:
        slope, _, _, _, _ = linregress(np.arange(5), gr_colors)
        color_slope_gr = slope
    except Exception:
        color_slope_gr = 0.0

    t_fade_pt = peak_time + (fade_time/2)
    gr_fade = -2.5 * np.log10(get_val(t_fade_pt, 'g') / get_val(t_fade_pt, 'r'))
    color_cooling_rate = gr_fade - gr_peak 
    
    def get_area(band):
        y_band = gp.predict(np.column_stack([t_grid, np.full_like(t_grid, FILTER_WAVELENGTHS[band])])) * y_scale
        return np.trapezoid(y_band, t_grid)
    
    total_area = get_area('u') + get_area('g') + get_area('r') + get_area('i')
    
    rise_fade_ratio = rise_time / fade_time if fade_time > 0 else 0
    area_under_curve = np.trapezoid(y_pred_g, t_grid)
    compactness = area_under_curve / peak_flux if peak_flux > 0 else 0
    rise_slope = amplitude / rise_time if rise_time > 1 else amplitude
    
    baseline_window = int(len(y_pred_g) * 0.15)
    baseline_flux = np.median(y_pred_g[:baseline_window])
    baseline_ratio = baseline_flux / peak_flux if peak_flux > 0 else 0

    return {
        'ObjectId': obj_id,
        'amplitude': amplitude,
        'ls_time': ls_time,
        'ls_wave': ls_wave,
        'rise_time': rise_time,
        'fade_time': fade_time,
        'fwhm': fwhm,
        'rise_fade_ratio': rise_fade_ratio,
        'compactness': compactness,
        'rise_slope': rise_slope,
        
        # Physics
        'tde_power_law_error': tde_error,
        'template_chisq_tde': match_tde,
        'linear_decay_slope': linear_decay_slope, 
        
        'mean_color_gr': mean_color_gr,
        'std_color_gr': std_color_gr,
        
        'total_radiated_energy_proxy': total_area, 
        'color_monotonicity': color_monotonicity, 
        'negative_flux_fraction': negative_flux_fraction,
        'percentile_ratio_20_50': percentile_ratio_20_50, 
        'percentile_ratio_80_max': percentile_ratio_80_max, 
        'rise_fireball_error': rise_error,
        'pre_peak_var': pre_peak_var, 
        'reduced_chi_square': reduced_chi_square,
        'fade_shape_correlation': fade_shape,
        'baseline_ratio': baseline_ratio,
        'color_cooling_rate': color_cooling_rate,
        'color_slope_gr': color_slope_gr, 
        'ug_peak': ug_peak,
        'gr_peak': gr_peak,
        'ur_peak': ur_peak,
        'flux_ratio_ug': flux_ratio_ug,
        'flux_ratio_gr': flux_ratio_gr,
        'flux_kurtosis': flux_kurtosis,
        'flux_skew': flux_skew, 
        'robust_duration': robust_duration,
        'duty_cycle': duty_cycle
    }


In [77]:
# get_gp_features(64073152, obj)

In [37]:
from joblib import Parallel, delayed


def compute_all_features(df):
        
    def process_row(row):
        obj_df = pl.DataFrame({k: [v] for k, v in row.items()})
        return get_gp_features(row['ObjectId'], obj_df)
        
    
    results = Parallel(n_jobs=-1)(
        delayed(process_row)(row) for row in df.iter_rows(named=True)
    )
    
    
    features_df = pl.DataFrame(results)

    # Determine redshift key
    z_key = None
    if 'Z' in df.columns:
        z_key = 'Z'
    elif 'HOSTGAL_PHOTOZ' in df.columns:
        z_key = 'HOSTGAL_PHOTOZ'
    
    if z_key is not None:
        print("Merging Redshift & Calculating Luminosity...")
        
        # Join redshift into features_df
        features_df = features_df.join(
            df.select(['ObjectId', z_key]).rename({z_key: 'redshift'}),
            on='ObjectId',
            how='left'
        )
        
        features_df = features_df.with_columns([
            # Clip replaces negative/NaN-like redshifts (e.g. sentinel values like -9)
            pl.col('redshift').clip(lower_bound=0.0).fill_null(0.0).alias('safe_z')
        ]).with_columns([
            (pl.col('rise_time') / (1.0 + pl.col('safe_z'))).alias('rest_rise_time'),
            (pl.col('fade_time') / (1.0 + pl.col('safe_z'))).alias('rest_fade_time'),
            (pl.col('fwhm')      / (1.0 + pl.col('safe_z'))).alias('rest_fwhm'),
            
            (-2.5 * (pl.col('amplitude').clip(lower_bound=0.001).log(base=10))
                   - 5.0 * (pl.col('safe_z') + 0.001).log(base=10)
            ).alias('absolute_magnitude_proxy'),
            
            (pl.col('total_radiated_energy_proxy') * (pl.col('safe_z') + 0.001) ** 2
            ).alias('total_radiated_energy'),
            
            ((pl.col('tde_power_law_error') + 1e-9).log(base=10)
            ).alias('log_tde_error'),
        ]).drop('safe_z')  # drop intermediate column if you don't want it


    df = df.join(features_df, on='ObjectId', how='left')
    return df, features_df

In [78]:
t = time.time()
parq_df_features, features_only = compute_all_features(parq)
print(time.time()-t)

/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/tmp/ipykernel

Merging Redshift & Calculating Luminosity...
33.982892751693726


/tmp/ipykernel_82692/2867331738.py:211: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.


In [79]:
features_only['linear_decay_slope']

linear_decay_slope
f64
0.0
0.0
0.0
0.0
0.0
…
0.0
0.0
0.0


In [80]:

features_pd = parq_df_features[['ObjectId', 'binary_class']].join(features_only, on='ObjectId', how='left').to_pandas()

In [90]:

X = features_pd.drop(columns=['ObjectId', 'binary_class'])
y = features_pd['binary_class']

In [93]:
# Check ALL types of missing
print("np.nan count:", np.isnan(X.values.astype(float)).sum())
print("np.inf count:", np.isinf(X.values.astype(float)).sum())
print("pd.NA / isnull count:", X.isnull().sum().sum())

# Find exactly which columns and how many
missing = X.isnull().sum()
print(missing[missing > 0])

# Force convert everything to float64 numpy (strips pd.NA)
X = X.astype(np.float64)
print("After astype - isnan:", np.isnan(X.values).sum())

np.nan count: 0
np.inf count: 0
pd.NA / isnull count: 0
Series([], dtype: int64)
After astype - isnan: 0


In [113]:

X = features_pd.drop(columns=['ObjectId', 'binary_class'])
y = features_pd['binary_class']

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0)
X_arr = selector.fit_transform(X)
kept_columns = X.columns[selector.get_support()]
dropped_columns = X.columns[~selector.get_support()]
print("Dropped constant columns:", dropped_columns.tolist())
X = pd.DataFrame(X_arr, columns=kept_columns)

X = X.replace([np.inf, -np.inf], np.nan)

Dropped constant columns: ['linear_decay_slope']


/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1872: RuntimeWarning: overflow encountered in multiply
  sqr = np.multiply(arr, arr, out=arr, where=where)


In [114]:
print("NaNs:", X.isna().sum().sum())
print("Infs:", np.isinf(X.values).sum())
print("\nInf columns:", X.columns[np.isinf(X.values).any(axis=0)].tolist())

NaNs: 0
Infs: 0

Inf columns: []


In [96]:
## Full code from https://github.com/maymeridian/Tidal-Disruption-Classification/blob/competition_winner/src/machine_learning/model_factory.py

'''
src/machine_learning/model_factory.py
Description: Hybrid Ensemble Classifier.
'''

import numpy as np
import json
import os
from catboost import CatBoostClassifier 
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.base import BaseEstimator, ClassifierMixin



# Root directory of the project
BASE_DIR = './lead1st_mallorn/'


MODEL_CONFIG = {
    'default_model': 'catboost',
    'random_seed': 67,
    'test_size': 0.2
}
MODELS_DIR = os.path.join(BASE_DIR, 'models')


# FEATURE SUBSETS

# Subset of Morphology & Temporal Evolution Features
MORPHOLOGY_FEATURES = [
    'rest_rise_time', 
    'rest_fade_time', 
    'rest_fwhm', 
    'ls_time',             
    'rise_fade_ratio', 
    'compactness', 
    'rise_slope', 
    'flux_kurtosis', 
    'robust_duration', 
    'duty_cycle', 
    'pre_peak_var', 
    'amplitude'
]

# Subset of Physics & Color Metrics
PHYSICS_FEATURES = [
    'tde_power_law_error', 
    'template_chisq_tde',
    'linear_decay_slope',
    'mean_color_gr',
    'std_color_gr',
    'total_radiated_energy',
    'color_monotonicity',
    'negative_flux_fraction',
    'rise_fireball_error', 
    'reduced_chi_square', 
    'ls_wave',             
    'fade_shape_correlation', 
    'baseline_ratio', 
    'color_cooling_rate', 
    'color_slope_gr', 
    'flux_ratio_ug', 
    'flux_ratio_gr', 
    'ug_peak', 'gr_peak', 'ur_peak', 
    'redshift', 
    'absolute_magnitude_proxy', 
    'log_tde_error'
]

class EnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, scale_pos_weight=1.0):
        self.scale_pos_weight = scale_pos_weight
        self.models = {} 
        self.feature_importances_ = None

    def fit(self, X, y):
        seed = MODEL_CONFIG['random_seed']
        
        # Gradient Boosting Parameters
        cb_params = {
            'iterations': 1000, 'depth': 5, 'learning_rate': 0.02,
            'l2_leaf_reg': 10, 'rsm': 0.5, 'loss_function': 'Logloss',
            'verbose': 0, 'allow_writing_files': False,
            'random_seed': seed, 'scale_pos_weight': self.scale_pos_weight
        }
        
        # Load optimized hyperparameters if available
        json_path = os.path.join(MODELS_DIR, 'best_params.json')
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f: 
                    tuned = json.load(f)
                if 'scale_pos_weight' in tuned: 
                    del tuned['scale_pos_weight']
                cb_params.update(tuned)
            except Exception: 
                pass

        # Base Model : CatBoost
        self.models['base'] = CatBoostClassifier(**cb_params)
        self.models['base'].fit(X, y)
        self.feature_importances_ = self.models['base'].feature_importances_

        # Morphology Specialist Model (CatBoost on only Shape Features)
        cols_morph = [c for c in MORPHOLOGY_FEATURES if c in X.columns]
        if cols_morph:
            self.models['morphology'] = CatBoostClassifier(**cb_params)
            self.models['morphology'].fit(X[cols_morph], y)

        # Physics Specialist (CatBoost on only Physics Features)
        cols_phys = [c for c in PHYSICS_FEATURES if c in X.columns]
        if cols_phys:
            self.models['physics'] = CatBoostClassifier(**cb_params)
            self.models['physics'].fit(X[cols_phys], y)

        # Supporting Model A: Multi-Layer Perceptron (Neural Network)
        self.models['mlp'] = Pipeline([
            ('var_filter', VarianceThreshold(threshold=0.0)),  # drop within-fold constants
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('net', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', 
                                  solver='adam', alpha=0.01, max_iter=600, 
                                  random_state=seed))
        ])
        self.models['mlp'].fit(X, y)

        # Supporting Model B: K-Nearest Neighbors
        self.models['knn'] = Pipeline([
            ('var_filter', VarianceThreshold(threshold=0.0)),
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('knn', KNeighborsClassifier(n_neighbors=15, weights='distance', p=2))
        ])
        self.models['knn'].fit(X, y)
        
        return self

    def predict_proba(self, X):
        # Generate Component Predictions
        p_base = self.models['base'].predict_proba(X)[:, 1]
        
        p_morph = p_base # Fallback
        if 'morphology' in self.models:
            cols = [c for c in MORPHOLOGY_FEATURES if c in X.columns]
            p_morph = self.models['morphology'].predict_proba(X[cols])[:, 1]
            
        p_phys = p_base # Fallback
        if 'physics' in self.models:
            cols = [c for c in PHYSICS_FEATURES if c in X.columns]
            p_phys = self.models['physics'].predict_proba(X[cols])[:, 1]
            
        p_mlp = self.models['mlp'].predict_proba(X)[:, 1]
        p_knn = self.models['knn'].predict_proba(X)[:, 1]
        
        # Ensemble Aggregation (Weighted Average)
        # Weights logic: 
        # - Gradient Boosting (80% Total): Split 48% Base, 16% Morph, 16% Phys
        # - Support Models (20% Total): Split 10% MLP, 10% KNN
        
        final_prob = (0.48 * p_base) + \
                     (0.16 * p_morph) + \
                     (0.16 * p_phys) + \
                     (0.10 * p_mlp) + \
                     (0.10 * p_knn)
        
        return np.vstack([1 - final_prob, final_prob]).T

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= 0.5).astype(int)

# FACTORY
def get_model(model_name, scale_pos_weight=1.0):
    if model_name == 'catboost':
        return EnsembleClassifier(scale_pos_weight=scale_pos_weight)
    else:
        raise ValueError("Only 'catboost' is supported.")

def train_with_cv(model_name, X, y):
    print("\n--- Initializing Hybrid Ensemble Classifier ---")
    print("    Configuration: Gradient Boosting (Core) + MLP/KNN (Support)")
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=MODEL_CONFIG['random_seed'])
    cv_scores = []
    best_thresholds = []

    fold = 1
    for train_index, val_index in skf.split(X, y):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        
        n_pos = y_train.sum()
        scale_weight = (len(y_train) - n_pos) / n_pos if n_pos > 0 else 1.0

        model = get_model(model_name, scale_pos_weight=scale_weight)
        model.fit(X_train, y_train)
        
        probs_val = model.predict_proba(X_val)[:, 1]
        
        best_f1 = 0.0
        best_t = 0.5
        for t in np.arange(0.2, 0.8, 0.02):
            s = f1_score(y_val, (probs_val >= t).astype(int), zero_division=0) 
            if s > best_f1: 
                best_f1 = s
                best_t = t
        
        val_tdes = y_val.sum()
        print(f"   Fold {fold}: F1={best_f1:.4f} (Thresh={best_t:.2f}) - Validation Positives: {val_tdes}")
        
        cv_scores.append(best_f1)
        best_thresholds.append(best_t)
        fold += 1

    avg_f1 = np.mean(cv_scores)
    avg_thresh = np.mean(best_thresholds)
    
    print(f"\n   Average Ensemble F1: {avg_f1:.4f}")
    
    # Final Train
    print("\n--- Training Final Production Model ---")
    n_pos_all = y.sum()
    final_weight = (len(y) - n_pos_all) / n_pos_all
    
    final_model = get_model(model_name, scale_pos_weight=final_weight)
    final_model.fit(X, y)
    
    return final_model, avg_f1, avg_thresh

In [100]:
print("Max abs value per column:")
print(X.abs().max().sort_values(ascending=False).head(10))

##wtf

Max abs value per column:
flux_ratio_ug                  5.637303e+233
tde_power_law_error            1.049377e+101
compactness                     4.137761e+90
flux_ratio_gr                   3.231377e+37
rise_fireball_error             2.254522e+21
pre_peak_var                    2.254522e+21
total_radiated_energy_proxy     1.517098e+07
total_radiated_energy           2.899832e+06
ls_wave                         1.000000e+05
ls_time                         1.000000e+05
dtype: float64


In [115]:
# Clip extreme values before passing to model
X = X.clip(lower=-1e10, upper=1e10)

In [116]:
### Probably would be more fair to merge train and val to handle the K-fold!

model, score, threshold = train_with_cv('catboost', X, y)


--- Initializing Hybrid Ensemble Classifier ---
    Configuration: Gradient Boosting (Core) + MLP/KNN (Support)
   Fold 1: F1=0.7200 (Thresh=0.56) - Validation Positives: 16
   Fold 2: F1=0.9091 (Thresh=0.54) - Validation Positives: 16
   Fold 3: F1=0.9333 (Thresh=0.36) - Validation Positives: 15
   Fold 4: F1=0.7368 (Thresh=0.40) - Validation Positives: 16
   Fold 5: F1=0.7222 (Thresh=0.20) - Validation Positives: 16

   Average Ensemble F1: 0.8043

--- Training Final Production Model ---


In [104]:
parq_ela_test = pl.read_parquet('/scratch/gcontard/ELASTICC2/mallorn_like/test_ELAsTiCC_10000ex_5.0percentTDE.parquet')

In [ ]:
t = time.time()
parq_df_features_test, features_only_test = compute_all_features(parq_ela_test)


/tmp/ipykernel_82692/2867331738.py:211: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
/tmp/ipykernel_82692/2867331738.py:41: RuntimeWarning: invalid value encountered in log
/tmp/ipykernel_82692/2867331738.py:41: RuntimeWarning: invalid value encountered in log
/tmp/ipykernel_82692/2867331738.py:41: RuntimeWarning: invalid value encountered in log
/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/u/g/gcontard/miniconda3/envs/py311/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fi

In [118]:
from sklearn.feature_selection import VarianceThreshold
print(time.time()-t)
features_pd_test = parq_df_features_test[['ObjectId', 'binary_class']].join(features_only_test, on='ObjectId', how='left').to_pandas()
X_test = features_pd_test.drop(columns=['ObjectId', 'binary_class'])
y_test = features_pd_test['binary_class']

X_test = X_test.clip(lower=-1e10, upper=1e10)

selector = VarianceThreshold(threshold=0.0)
X_arr = selector.fit_transform(X_test)
kept_columns = X_test.columns[selector.get_support()]
dropped_columns = X_test.columns[~selector.get_support()]
print("Dropped constant columns:", dropped_columns.tolist())

X_test = pd.DataFrame(X_arr, columns=kept_columns)

X_test = X_test.replace([np.inf, -np.inf], np.nan)

55573.87797808647
Dropped constant columns: ['linear_decay_slope']


In [119]:
dropped_columns

Index(['linear_decay_slope'], dtype='object')

In [120]:
pred_test = model.predict(X_test)

In [121]:
from sklearn.metrics import classification_report

In [124]:
print(classification_report(y_test, pred_test))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9500
           1       0.85      0.77      0.81       500

    accuracy                           0.98     10000
   macro avg       0.92      0.88      0.90     10000
weighted avg       0.98      0.98      0.98     10000

